In [6]:
# System and core libraries
import sys
from pathlib import Path
import time
from datetime import datetime

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Data processing
import pandas as pd
import numpy as np
import json

# TactIQ modules
from src.embeddings import EmbeddingPipeline
from src.database import VectorDatabase
from src.chunking_strategy import create_chunking_strategy, validate_chunk_quality

# Logging
from loguru import logger

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(" TACTIQ DAY 2 - EMBEDDING & VECTOR DATABASE SETUP")

print(f" Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f" Python: {sys.version.split()[0]}")
print(f" Project Root: {PROJECT_ROOT}")


 TACTIQ DAY 2 - EMBEDDING & VECTOR DATABASE SETUP
 Start Time: 2026-01-05 13:36:28
 Python: 3.10.19
 Project Root: C:\Users\Hp\Frost Digital Ventures\TactIQ


#  TactIQ - Day 2: Embedding & Vector Database Setup

**Project**: Multi-Step RAG for Football Scouting  
**Date**: December 30, 2025 (Day 2)  
**Status**: Data Collection Complete -> Embedding & VectorDB Setup 


## Day 2 Objectives:

1.  Load cleaned player statistics from Day 1
2.  Convert structured tables to natural language descriptions
3.  Deduplicate records (remove 5,500 duplicates)
4.  Initialize all-MiniLM-L6-v2 embedding model
5.  Set up ChromaDB vector database
6.  Embed and ingest player descriptions (batch processing)
7.  Validate semantic search and retrieval quality
8.  Generate Day 2 completion report


##  Expected Outcomes:

- **Input**: 19,388 raw player records
- **After Dedup**: ~13,900 unique player-season-team-league records 
- **Output**: ChromaDB with 13.9K+ embedded player profiles
- **Storage**: ~30-50 MB vector database
- **Chunking**: Hybrid Entity-Aligned + Semantic strategy


##  Load Cleaned Player Data from Day 1

In [7]:
#  LOAD CLEANED DATA FROM DAY 1
print( "LOADING CLEANED PLAYER DATA")

# Define data paths
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
CLEANED_CSV = PROCESSED_DIR / 'player_stats_unified_CLEANED.csv'

print(f"\n Data Source:")
print(f"   File: {CLEANED_CSV.name}")
print(f"   Path: {CLEANED_CSV}")

# Check if file exists
if not CLEANED_CSV.exists():
    raise FileNotFoundError(f" Cleaned data not found: {CLEANED_CSV}")

# Load data
print(f"\n Loading data...")
start_load = time.time()
df_cleaned = pd.read_csv(CLEANED_CSV, low_memory=False)
load_time = time.time() - start_load

print(f"\n Data Loaded Successfully!")
print(f"   Time: {load_time:.2f}s")
print(f"   Shape: {df_cleaned.shape[0]:,} rows × {df_cleaned.shape[1]} columns")
print(f"   Memory: {df_cleaned.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display sample
print(f"\n Data Preview:")
print(df_cleaned.head(3))

# Key columns check
key_cols = ['player', 'team', 'league', 'season', 'pos', 'age']
available_cols = [col for col in key_cols if col in df_cleaned.columns]
print(f"\n Key Columns Present: {', '.join(available_cols)}")

missing_cols = [col for col in key_cols if col not in df_cleaned.columns]
if missing_cols:
    print(f"  Missing Columns: {', '.join(missing_cols)}")

# Clean age column (remove concatenated values like "23-171")
if 'age' in df_cleaned.columns:
    df_cleaned['age'] = df_cleaned['age'].astype(str).str.split('-').str[0]
    df_cleaned['age'] = pd.to_numeric(df_cleaned['age'], errors='coerce').fillna(0).astype(int)
    print(f"\n Age column cleaned (removed concatenated values)")

LOADING CLEANED PLAYER DATA

 Data Source:
   File: player_stats_unified_CLEANED.csv
   Path: C:\Users\Hp\Frost Digital Ventures\TactIQ\data\processed\player_stats_unified_CLEANED.csv

 Loading data...



 Data Loaded Successfully!
   Time: 2.47s
   Shape: 19,388 rows × 120 columns
   Memory: 29.63 MB

 Data Preview:
               league     team              player nation pos     age    born  \
0  ENG-Premier League  Arsenal           Ben White    ENG  DF  28-082  1997.0   
1  ENG-Premier League  Arsenal         Bukayo Saka    ENG  FW  24-115  2001.0   
2  ENG-Premier League  Arsenal  Christian Nørgaard    DEN  MF  31-294  1994.0   

   Playing Time_MP_std  Playing Time_Starts_std  Playing Time_Min_std  \
0                    4                        4                   280   
1                   16                       14                  1239   
2                    2                        0                    15   

   Playing Time_90s_std  Performance_Gls_std  Performance_Ast_std  \
0                   3.1                    0                    1   
1                  13.8                    4                    2   
2                   0.2                    0                

##  Table-to-Text Conversion: Natural Language Descriptions

In [ ]:
#  TABLE-TO-TEXT CONVERTER & DEDUPLICATION
print(" CONVERTING PLAYER STATS TO NATURAL LANGUAGE")

def create_player_description(row):
    """Convert player stats row to semantic-rich natural language description"""
    parts = []
    
    # Identity & Demographics
    player = row.get('player', 'Unknown Player')
    age = row.get('age', 'unknown')
    pos = row.get('pos', 'unknown position')
    team = row.get('team', 'unknown team')
    league = row.get('league', 'unknown league')
    season = row.get('season', 'unknown season')
    
    parts.append(
        f"{player} is a {age}-year-old {pos} playing for {team} in {league} during the {season} season."
    )
    
    # Performance Stats
    goals = row.get('Performance_Gls_std', 0)
    assists = row.get('Performance_Ast_std', 0)
    minutes = row.get('Playing Time_Min_std', 0)
    matches = row.get('Playing Time_MP_std', 0)
    
    if pd.notna(goals) and goals > 0:
        parts.append(
            f"Performance: {int(goals)} goals and {int(assists)} assists in {int(matches)} matches ({int(minutes)} minutes)."
        )
    
    # Expected Stats
    xg = row.get('Expected_xG_std', 0)
    xag = row.get('Expected_xAG_std', 0)
    if pd.notna(xg) and xg > 0:
        parts.append(f"Expected performance: {xg:.1f} xG and {xag:.1f} xAG.")
    
    # Shooting
    shots = row.get('Standard_Sh_shoot', 0)
    sot = row.get('Standard_SoT_shoot', 0)
    if pd.notna(shots) and shots > 0:
        parts.append(f"Shooting: {int(shots)} shots with {int(sot)} on target.")
    
    # Passing
    passes_cmp = row.get('Total_Cmp_pass', 0)
    passes_att = row.get('Total_Att_pass', 0)
    if pd.notna(passes_cmp) and passes_att > 0:
        pass_pct = (passes_cmp / passes_att * 100)
        parts.append(f"Passing: {int(passes_cmp)}/{int(passes_att)} completed ({pass_pct:.1f}% accuracy).")
    
    # Defensive
    tackles = row.get('Tackles_Tkl_def', 0)
    interceptions = row.get('Int_def', 0)
    if pd.notna(tackles) and tackles > 0:
        parts.append(f"Defensive: {int(tackles)} tackles and {int(interceptions)} interceptions.")
    
    # Market Value
    market_value = row.get('market_value', None)
    if pd.notna(market_value):
        parts.append(f"Market value: €{market_value}M.")
    
    return " ".join(parts)

# Generate descriptions
print(f"\n Generating descriptions for {len(df_cleaned):,} players...")
start_conv = time.time()

df_cleaned['description'] = df_cleaned.apply(create_player_description, axis=1)

conv_time = time.time() - start_conv

print(f"\n Conversion Complete!")
print(f"   Time: {conv_time:.2f}s")
print(f"   Speed: {len(df_cleaned)/conv_time:.0f} descriptions/second")

# Show samples
print(f"\n Sample Descriptions:\n" + "-"*80)
for i in range(min(3, len(df_cleaned))):
    print(f"\n[{i+1}] {df_cleaned['player'].iloc[i]}:")
    print(f"    {df_cleaned['description'].iloc[i][:200]}...")

# Save descriptions
desc_file = PROCESSED_DIR / 'player_descriptions_for_embedding.csv'
df_cleaned[['player', 'team', 'league', 'season', 'description']].to_csv(desc_file, index=False)
print(f"\n Saved: {desc_file.name} ({desc_file.stat().st_size / 1024:.0f} KB)")

# DEDUPLICATION - Create df_final (deduplicated dataset)
print(" DEDUPLICATION - Creating df_final")

duplicate_cols = ['player', 'team', 'league', 'season']
before_count = len(df_cleaned)

# Remove duplicates and create df_final
df_final = df_cleaned.drop_duplicates(
    subset=duplicate_cols,
    keep='first'
).reset_index(drop=True)

after_count = len(df_final)
removed = before_count - after_count

print(f"\n   Before: {before_count:,} rows")
print(f"   After: {after_count:,} rows")
print(f"   Removed: {removed:,} duplicates")

# Final validation
dup_check = df_final[df_final.duplicated(subset=duplicate_cols, keep=False)]

print(f"\n FINAL DATASET VALIDATION:")
print(f" Total records: {len(df_final):,}")
print(f" Duplicate IDs: {len(dup_check)}")
print(f" Status: {' CLEAN (ready for chunking)' if len(dup_check) == 0 else ' DUPLICATES FOUND'}")
print(f" Columns: {', '.join(df_final.columns.tolist()[:5])}... (+{len(df_final.columns)-5} more)")
print(f" Memory: {df_final.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print(f"\n df_final ({len(df_final):,} deduplicated records) READY FOR CHUNKING")

 CONVERTING PLAYER STATS TO NATURAL LANGUAGE

 Generating descriptions for 19,388 players...



 Conversion Complete!
   Time: 6.23s
   Speed: 3111 descriptions/second

 Sample Descriptions:
--------------------------------------------------------------------------------

[1] Ben White:
    Ben White is a 28-year-old DF playing for Arsenal in ENG-Premier League during the 2025-2026 season. Expected performance: 0.4 xG and 0.7 xAG. Shooting: 3 shots with 1 on target. Passing: 125/159 comp...

[2] Bukayo Saka:
    Bukayo Saka is a 24-year-old FW playing for Arsenal in ENG-Premier League during the 2025-2026 season. Performance: 4 goals and 2 assists in 16 matches (1239 minutes). Expected performance: 4.8 xG and...

[3] Christian Nørgaard:
    Christian Nørgaard is a 31-year-old MF playing for Arsenal in ENG-Premier League during the 2025-2026 season. Expected performance: 0.1 xG and 0.0 xAG. Shooting: 1 shots with 1 on target. Passing: 5/7...

 Saved: player_descriptions_for_embedding.csv (6351 KB)
 DEDUPLICATION - Creating df_final

   Before: 19,388 rows
   After: 13,888 rows
  

In [9]:
# DEDUPLICATE FINAL DATAFRAME
print("DEDUPLICATION: Creating Final Clean Dataframe")


# Check for duplicates
duplicate_cols = ['player', 'team', 'league', 'season']
before_dedup = len(df_cleaned)

print(f"\nBefore deduplication: {before_dedup:,} records")

# Find duplicates
duplicates = df_cleaned[df_cleaned.duplicated(subset=duplicate_cols, keep=False)]
if len(duplicates) > 0:
    print(f"Found {len(duplicates)} duplicate records")
    print(f"\nSample duplicates:")
    print(duplicates[duplicate_cols].head(10))

# Deduplicate: keep first occurrence of each player-season-team-league combo
df_final = df_cleaned.drop_duplicates(
    subset=duplicate_cols,
    keep='first'
).reset_index(drop=True)

after_dedup = len(df_final)
removed = before_dedup - after_dedup

print(f"\nDeduplication Complete!")
print(f" Before: {before_dedup:,} records")
print(f" After:  {after_dedup:,} records")
print(f" Removed: {removed:,} duplicates ({100*removed/before_dedup:.1f}%)")

# Try to save deduplicated final dataframe (skip if file is locked)
final_df_path = PROCESSED_DIR / 'player_stats_unified_FINAL_DEDUPED.csv'
try:
    df_final.to_csv(final_df_path, index=False)
    print(f"\n Saved final deduplicated dataframe:")
    print(f"   File: {final_df_path.name}")
    print(f"   Size: {final_df_path.stat().st_size / 1024:.0f} KB")
    print(f"   Records: {len(df_final):,}")
except PermissionError:
    print(f"\n Could not save CSV (file is locked - likely open in Excel)")
    print(f"   File: {final_df_path.name}")
    print(f"   Solution: Close Excel and re-run, or continue without saving")
    print(f"    df_final is ready in memory for ingestion")

print(f"\n FINAL DATASET READY FOR EMBEDDING:")
print(f"   Total unique player-season-team-league records: {len(df_final):,}")
print(f"   No duplicates ")
print(f"   All records have descriptions ")
print(f"   Assigned to df_final for downstream use ")

DEDUPLICATION: Creating Final Clean Dataframe

Before deduplication: 19,388 records
Found 6256 duplicate records

Sample duplicates:
            player         team              league     season
20  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
21  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
22  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
23  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
24  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
25  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
26  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
27  Piero Hincapié      Arsenal  ENG-Premier League  2025-2026
39  Harvey Elliott  Aston Villa  ENG-Premier League  2025-2026
40  Harvey Elliott  Aston Villa  ENG-Premier League  2025-2026

Deduplication Complete!
 Before: 19,388 records
 After:  13,888 records
 Removed: 5,500 duplicates (28.4%)

 Saved final deduplicated dataframe:
   File: player_stats_unifie

##  Initialize Embedding Model: all-MiniLM-L6-v2

In [10]:
#  INITIALIZE EMBEDDING MODEL
print(" INITIALIZING EMBEDDING MODEL")

print(f"\n Loading sentence-transformers model...")
start_model = time.time()

# Initialize embedding pipeline
embedding_pipeline = EmbeddingPipeline()

model_time = time.time() - start_model

print(f"\n Model Loaded!")
print(f" Model: {embedding_pipeline.model_name}")
print(f" Dimensions: {embedding_pipeline.model.get_sentence_embedding_dimension()}")
print(f" Max Sequence Length: {embedding_pipeline.model.max_seq_length}")
print(f" Load Time: {model_time:.2f}s")

# Test embedding
print(f"\n Testing embedding generation...")
sample_text = df_cleaned['description'].iloc[0]
print(f"Sample: {sample_text[:80]}...")

test_emb = embedding_pipeline.embed_text(sample_text)
print(f"\n Embedding Test Successful!")
print(f"Shape: {test_emb.shape}")
print(f"Type: {type(test_emb)}")
print(f"Sample values: {test_emb[:5]}")

2026-01-05 13:36:47.145 | INFO     | src.embeddings:__init__:34 - Loading embedding model: all-MiniLM-L6-v2


 INITIALIZING EMBEDDING MODEL

 Loading sentence-transformers model...


2026-01-05 13:37:13.022 | INFO     | src.embeddings:__init__:36 - Model loaded successfully. Embedding dimension: 384



 Model Loaded!
 Model: sentence-transformers/all-MiniLM-L6-v2
 Dimensions: 384
 Max Sequence Length: 256
 Load Time: 26.54s

 Testing embedding generation...
Sample: Ben White is a 28-year-old DF playing for Arsenal in ENG-Premier League during t...

 Embedding Test Successful!
Shape: (384,)
Type: <class 'numpy.ndarray'>
Sample values: [ 0.07831299  0.01779885 -0.02911655 -0.06918324  0.02992355]


##  Initialize ChromaDB Vector Database

In [11]:
#  INITIALIZE CHROMADB (FAST MODE)
print(" INITIALIZING CHROMADB VECTOR DATABASE")

# Database configuration
DB_PATH = PROJECT_ROOT / 'db' / 'chroma'
COLLECTION_NAME = 'player_stats'

print(f"\n Configuration:")
print(f"   Path: {DB_PATH}")
print(f"   Collection: {COLLECTION_NAME}")

# Create database directory
DB_PATH.mkdir(parents=True, exist_ok=True)

# Initialize vector database
print(f"\n Creating ChromaDB instance...")
vector_db = VectorDatabase(
    persist_directory=str(DB_PATH),
    collection_name=COLLECTION_NAME
)

print(f"\n ChromaDB Initialized.")
print(f"   Client: {type(vector_db.client).__name__}")
print(f"   Collection: {vector_db.collection_name}")

# Check existing documents (non-blocking)
try:
    existing_count = vector_db.collection.count()
    print(f"   Existing documents: {existing_count:,}")
    
    if existing_count > 0:
        print(f"    Using existing collection (no reset)")
        print(f"    To reset manually, run: vector_db.reset_collection()")
except Exception as e:
    print(f"   New collection created (no existing data)")
    print(f"   Error checking: {e}")



 INITIALIZING CHROMADB VECTOR DATABASE

 Configuration:
   Path: C:\Users\Hp\Frost Digital Ventures\TactIQ\db\chroma
   Collection: player_stats

 Creating ChromaDB instance...


2026-01-05 13:37:23.489 | INFO     | src.database:_get_or_create_collection:73 - Loaded existing collection: player_stats
2026-01-05 13:37:23.491 | INFO     | src.database:__init__:63 - Initialized ChromaDB at C:\Users\Hp\Frost Digital Ventures\TactIQ\db\chroma
2026-01-05 13:37:23.543 | INFO     | src.database:__init__:64 - Collection: player_stats with 0 documents



 ChromaDB Initialized.
   Client: Client
   Collection: player_stats
   Existing documents: 0


In [12]:
#  MODULAR CHUNKING: SEMANTIC STAT MODULES (PRODUCTION ARCHITECTURE)

print(" MODULAR CHUNKING STRATEGY")
print("Architecture: 4-6 semantic modules per player-season")
print("Inspired by: Wyscout, StatsBomb, Pro Analytics Stacks")
print("-"*80)

# Configuration
MAX_RECORDS = None  # None = all records

# Prepare data
print("\n Loading deduplicated data...")
if 'df_final' not in globals():
    raise ValueError(" df_final not found! Run deduplication cell first.")

base_df = df_final.copy()
if MAX_RECORDS:
    df_to_embed = base_df.head(MAX_RECORDS).copy()
    print(f" Testing with {MAX_RECORDS} records")
else:
    df_to_embed = base_df.copy()

print(f" Player-seasons: {len(df_to_embed):,}")
print(f"Total columns: {len(df_to_embed.columns)}")

# Define semantic modules (column groups)
STAT_MODULES = {
    'identity': {
        'name': 'Identity & Context',
        'description': 'Player profile, minutes, team context',
        'columns': ['player', 'age', 'pos', 'team', 'league', 'nation', 'born',
                   'competition', 'season',
                   'Playing Time_MP_std', 'Playing Time_Starts_std', 
                   'Playing Time_Min_std', 'Playing Time_90s_std',
                   'market_value'],
        'always_retrieve': True  # Always included
    },
    'shooting': {
        'name': 'Scoring & Shooting',
        'description': 'Goals, shots, finishing, xG profile',
        'columns': ['Performance_Gls_std', 'Performance_Ast_std', 'Performance_G+A_std',
                   'Performance_G-PK_std', 'Performance_PK_std', 'Performance_PKatt_std',
                   'Expected_xG_std', 'Expected_npxG_std', 'Expected_xAG_std',
                   'Expected_npxG+xAG_std',
                   '90s_shoot', 'Standard_Gls_shoot', 'Standard_Sh_shoot', 
                   'Standard_SoT_shoot', 'Standard_SoT%_shoot',
                   'Standard_Sh/90_shoot', 'Standard_SoT/90_shoot', 
                   'Standard_G/Sh_shoot', 'Standard_G/SoT_shoot', 
                   'Standard_Dist_shoot', 'Standard_FK_shoot',
                   'Standard_PK_shoot', 'Standard_PKatt_shoot',
                   'Expected_xG_shoot', 'Expected_npxG_shoot', 
                   'Expected_npxG/Sh_shoot', 'Expected_G-xG_shoot', 
                   'Expected_np:G-xG_shoot'],
        'query_keywords': ['goal', 'shot', 'finish', 'score', 'xG', 'striker', 'forward']
    },
    'passing': {
        'name': 'Creativity & Passing',
        'description': 'Assists, key passes, chance creation, pass completion',
        'columns': ['Expected_xAG_std', '90s_pass',
                   'Total_Cmp_pass', 'Total_Att_pass', 'Total_Cmp%_pass',
                   'Total_TotDist_pass', 'Total_PrgDist_pass',
                   'Short_Cmp_pass', 'Short_Att_pass', 'Short_Cmp%_pass',
                   'Medium_Cmp_pass', 'Medium_Att_pass', 'Medium_Cmp%_pass',
                   'Long_Cmp_pass', 'Long_Att_pass', 'Long_Cmp%_pass',
                   'Ast_pass', 'xAG_pass', 'Expected_xA_pass', 'Expected_A-xAG_pass', 
                   'KP_pass', '1/3_pass', 'PPA_pass', 'CrsPA_pass', 'PrgP_pass'],
        'query_keywords': ['pass', 'assist', 'creative', 'chance', 'playmaker', 'link', 'xAG', 'key pass']
    },
    'progression': {
        'name': 'Ball Progression & Carrying',
        'description': 'Progressive carries, passes, and receiving',
        'columns': ['Progression_PrgC_std', 'Progression_PrgP_std', 'Progression_PrgR_std',
                   'Total_PrgDist_pass', 'PrgP_pass'],
        'query_keywords': ['dribble', 'carry', 'progress', 'touch', 'beat', 'run', 'take-on', 'progressive']
    },
    'defensive': {
        'name': 'Defensive Contribution',
        'description': 'Tackles, pressures, interceptions, defensive actions',
        'columns': ['90s_def', 
                   'Tackles_Tkl_def', 'Tackles_TklW_def', 'Tackles_Def 3rd_def',
                   'Tackles_Mid 3rd_def', 'Tackles_Att 3rd_def',
                   'Challenges_Tkl_def', 'Challenges_Att_def', 'Challenges_Tkl%_def',
                   'Challenges_Lost_def', 'Blocks_Blocks_def', 'Blocks_Sh_def',
                   'Blocks_Pass_def', 'Int_def', 'Tkl+Int_def', 'Clr_def', 'Err_def'],
        'query_keywords': ['tackle', 'press', 'intercept', 'block', 'defend', 'win back', 'defensive']
    }
}

# GK-specific module (only for goalkeepers)
GK_MODULE = {
    'name': 'Goalkeeping',
    'description': 'Saves, distribution, sweeping, shot-stopping',
    'columns': ['Playing Time_MP_gk', 'Playing Time_Starts_gk', 
               'Playing Time_Min_gk', 'Playing Time_90s_gk',
               'Performance_GA_gk', 'Performance_GA90_gk', 'Performance_SoTA_gk',
               'Performance_Saves_gk', 'Performance_Save%_gk', 
               'Performance_W_gk', 'Performance_D_gk', 'Performance_L_gk', 
               'Performance_CS_gk', 'Performance_CS%_gk',
               'Penalty Kicks_PKatt_gk', 'Penalty Kicks_PKA_gk', 
               'Penalty Kicks_PKsv_gk', 'Penalty Kicks_PKm_gk'],
    'query_keywords': ['goalkeeper', 'save', 'clean sheet', 'distribution', 'GK']
}

# Create modular chunks
print(f"\n Creating semantic stat modules...")
chunk_start = time.time()

all_chunks = []
module_counts = {module: 0 for module in STAT_MODULES.keys()}
module_counts['goalkeeper'] = 0

for idx, row in df_to_embed.iterrows():
    player = row.get('player', "Unknown")
    season = row.get('season', "Unknown")
    team = row.get('team', row.get('Squad', "Unknown"))
    competition = row.get('league', row.get('Comp', "Unknown"))
    pos = row.get('pos', row.get('position', 'N/A'))
    
    # Entity ID (include team to avoid duplicates for mid-season transfers)
    entity_id = f"{player}_{team}_{season}_{competition}".replace(" ", "_").replace("/", "-")
    
    # Check if goalkeeper
    is_gk = 'GK' in str(pos).upper()
    
    # MODULE 1: Identity (ALWAYS created)
    identity_cols = STAT_MODULES['identity']['columns']
    identity_meta = {
        'chunk_id': f"player_{entity_id}_identity",
        'parent_id': f"player_{player.replace(' ', '_')}",
        'chunk_type': 'player_stats',
        'stat_module': 'identity',
        'player': player,
        'team': team,
        'season': season,
        'competition': competition,
        'pos': pos,
        'source': 'FBref'
    }
    
    # Add identity columns to metadata
    for col in identity_cols:
        if col in row.index:
            val = row[col]
            if pd.notna(val) and str(val).strip() != '':
                identity_meta[col] = val
    
    # Create identity content
    age = identity_meta.get('age', identity_meta.get('Age', 'N/A'))
    minutes = identity_meta.get('Playing Time_Min_std', identity_meta.get('minutes', 0))
    matches = identity_meta.get('Playing Time_MP_std', identity_meta.get('matches', 0))
    market_val = identity_meta.get('market_value', '')
    
    identity_content = (
        f"{player} is a {age}-year-old {pos} playing for {team} in {competition} "
        f"during the {season} season. Played {matches} matches ({minutes} minutes)."
    )
    if market_val:
        identity_content += f" Market value: €{market_val}M."
    
    all_chunks.append({
        'content': identity_content,
        'metadata': identity_meta,
        'id': identity_meta['chunk_id']
    })
    module_counts['identity'] += 1
    
    # MODULE 2-5: Stat modules (only if relevant data exists)
    for module_name, module_config in STAT_MODULES.items():
        if module_name == 'identity':
            continue  # Already created
        
        module_cols = module_config['columns']
        module_meta = {
            'chunk_id': f"player_{entity_id}_{module_name}",
            'parent_id': f"player_{player.replace(' ', '_')}",
            'chunk_type': 'player_stats',
            'stat_module': module_name,
            'player': player,
            'team': team,
            'season': season,
            'competition': competition,
            'pos': pos,
            'source': 'FBref'
        }
        
        # Add module-specific columns
        cols_found = 0
        for col in module_cols:
            if col in row.index:
                val = row[col]
                if pd.notna(val) and str(val).strip() != '':
                    module_meta[col] = val
                    cols_found += 1
        
        # Only create chunk if we have data
        if cols_found > 0:
            module_content = f"{player} ({season}) - {module_config['name']}: {module_config['description']}"
            
            all_chunks.append({
                'content': module_content,
                'metadata': module_meta,
                'id': module_meta['chunk_id']
            })
            module_counts[module_name] += 1
    
    # MODULE 6: Goalkeeper (only for GKs)
    if is_gk:
        gk_cols = GK_MODULE['columns']
        gk_meta = {
            'chunk_id': f"player_{entity_id}_goalkeeper",
            'parent_id': f"player_{player.replace(' ', '_')}",
            'chunk_type': 'player_stats',
            'stat_module': 'goalkeeper',
            'player': player,
            'team': team,
            'season': season,
            'competition': competition,
            'pos': pos,
            'source': 'FBref'
        }
        
        # Add GK columns
        gk_cols_found = 0
        for col in gk_cols:
            if col in row.index:
                val = row[col]
                if pd.notna(val) and str(val).strip() != '':
                    gk_meta[col] = val
                    gk_cols_found += 1
        
        if gk_cols_found > 0:
            gk_content = f"{player} ({season}) - Goalkeeping: Saves, distribution, shot-stopping"
            
            all_chunks.append({
                'content': gk_content,
                'metadata': gk_meta,
                'id': gk_meta['chunk_id']
            })
            module_counts['goalkeeper'] += 1

chunk_time = time.time() - chunk_start

print(f"\nMODULAR CHUNKING COMPLETE!")
print(f" Player-seasons: {len(df_to_embed):,}")
print(f" Total chunks: {len(all_chunks):,}")
print(f" Time: {chunk_time:.2f}s")
print(f" Ratio: {len(all_chunks)/len(df_to_embed):.2f} chunks per player")
print(f"\n Chunks by Module:")
for module, count in module_counts.items():
    print(f"   {module:15} → {count:,}")

# Show sample chunks
print(f"\n Sample Modular Chunks (First Player):")
sample_player = df_to_embed.iloc[0]['player']
player_chunks = [c for c in all_chunks if c['metadata']['player'] == sample_player][:4]
for i, chunk in enumerate(player_chunks, 1):
    meta = chunk['metadata']
    print(f"\n[{i}] Module: {meta['stat_module'].upper()}")
    print(f" Chunk ID: {meta['chunk_id']}")
    print(f" Metadata fields: {len(meta)}")
    print(f" Content: {chunk['content'][:100]}...")

# Prepare for ChromaDB
documents = [chunk['content'] for chunk in all_chunks]
metadatas = [chunk['metadata'] for chunk in all_chunks]
ids = [chunk['id'] for chunk in all_chunks]

print(f"\n Ingesting into ChromaDB...")
print(f" Batch size: 500")

# Batch processing
BATCH_SIZE = 500
total_records = len(documents)
num_batches = (total_records + BATCH_SIZE - 1) // BATCH_SIZE

print(f" Total batches: {num_batches}")

total_embedded = 0
start_embed = time.time()

for batch_idx in range(0, total_records, BATCH_SIZE):
    batch_num = batch_idx // BATCH_SIZE + 1
    batch_end = min(batch_idx + BATCH_SIZE, total_records)
    
    batch_docs = documents[batch_idx:batch_end]
    batch_metas = metadatas[batch_idx:batch_end]
    batch_ids = ids[batch_idx:batch_end]
    
    batch_start = time.time()
    vector_db.add_documents(
        documents=batch_docs,
        metadatas=batch_metas,
        ids=batch_ids
    )
    batch_time = time.time() - batch_start
    total_embedded += len(batch_docs)
    
    if batch_num % 10 == 0 or batch_num == num_batches:
        pct = (total_embedded / total_records) * 100
        print(f" Batch {batch_num}/{num_batches}: {len(batch_docs)} docs | {pct:.1f}%")

embed_time = time.time() - start_embed

print(f"\n MODULAR INGESTION COMPLETE!")
print(f"  Total chunks: {total_embedded:,}")
print(f"  Time: {embed_time:.2f}s ({embed_time/60:.2f} min)")
print(f"  Speed: {total_embedded/embed_time:.1f} docs/s")
print(f"\ Architecture: Production-grade semantic modules")
print(f" Query-aware retrieval ready")
print(f" No hallucination - all stats explicit")

 MODULAR CHUNKING STRATEGY
Architecture: 4-6 semantic modules per player-season
Inspired by: Wyscout, StatsBomb, Pro Analytics Stacks
--------------------------------------------------------------------------------

 Loading deduplicated data...
 Player-seasons: 13,888
Total columns: 121

 Creating semantic stat modules...

MODULAR CHUNKING COMPLETE!
 Player-seasons: 13,888
 Total chunks: 70,279
 Time: 13.35s
 Ratio: 5.06 chunks per player

 Chunks by Module:
   identity        → 13,888
   shooting        → 13,888
   passing         → 13,888
   progression     → 13,887
   defensive       → 13,888
   goalkeeper      → 840

 Sample Modular Chunks (First Player):

[1] Module: IDENTITY
 Chunk ID: player_Ben_White_Arsenal_2025-2026_ENG-Premier_League_identity
 Metadata fields: 19
 Content: Ben White is a 28-year-old DF playing for Arsenal in ENG-Premier League during the 2025-2026 season....

[2] Module: SHOOTING
 Chunk ID: player_Ben_White_Arsenal_2025-2026_ENG-Premier_League_shooting
 Met

2026-01-05 13:37:58.585 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:06.903 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:14.203 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:20.241 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:26.745 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:32.948 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:39.384 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:46.227 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:38:54.085 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:01.451 | INFO     | src.database:add_documents:

 Batch 10/141: 500 docs | 7.1%


2026-01-05 13:39:06.732 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:12.506 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:18.112 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:23.316 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:28.433 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:33.785 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:40.831 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:48.923 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:39:55.878 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:40:02.168 | INFO     | src.database:add_documents:

 Batch 20/141: 500 docs | 14.2%


...

 Batch 130/141: 500 docs | 92.5%


2026-01-05 13:52:32.186 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:52:37.988 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:52:44.843 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:52:52.060 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:53:01.891 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:53:11.436 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:53:19.227 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:53:27.717 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:53:35.525 | INFO     | src.database:add_documents:110 - Added 500 documents to collection
2026-01-05 13:53:43.882 | INFO     | src.database:add_documents:

 Batch 140/141: 500 docs | 99.6%


2026-01-05 13:53:49.531 | INFO     | src.database:add_documents:110 - Added 279 documents to collection


 Batch 141/141: 279 docs | 100.0%

 MODULAR INGESTION COMPLETE!
  Total chunks: 70,279
  Time: 965.57s (16.09 min)
  Speed: 72.8 docs/s
\ Architecture: Production-grade semantic modules
 Query-aware retrieval ready
 No hallucination - all stats explicit


In [ ]:
# Initialize blog scraper
from script.data_collection.blog_scraper import BlogScraper

scraper = BlogScraper(data_dir=str(BLOG_DATA_DIR))
print(" Blog scraper initialized")

# Track scraped URLs to avoid duplicates
scraped_urls = {art.get('url') for art in existing_articles if art.get('url')}
print(f" Tracking {len(scraped_urls)} existing URLs to avoid duplicates")

2025-12-30 16:49:14.916 | INFO     | script.data_collection.blog_scraper:__init__:60 - Initialized blog scraper with output dir: ..\data\blogs


 Blog scraper initialized
 Tracking 36 existing URLs to avoid duplicates


In [ ]:
# STEP 1: LOAD BLOG ARTICLES
print(" LOADING TACTICAL BLOG ARTICLES")

blog_file = PROJECT_ROOT / "data" / "blogs" / "tactical_blogs_20251229_202542.json"

if blog_file.exists():
    with open(blog_file, 'r', encoding='utf-8') as f:
        blog_articles = json.load(f)
    
    print(f"\nLoaded {len(blog_articles)} tactical blog articles")
    
    if blog_articles:
        sample = blog_articles[0]
        print(f"\nSample Article:")
        print(f"  Title: {sample.get('title', 'N/A')[:70]}...")
        print(f"  Source: {sample.get('source', 'N/A')}")
        print(f"  Text Length: {len(sample.get('text', '')):,} characters")
        
        total_chars = sum(len(art.get('text', '')) for art in blog_articles)
        print(f"\nBlog Statistics:")
        print(f"  Total Articles: {len(blog_articles)}")
        print(f"  Total Characters: {total_chars:,}")
        print(f"  Avg Length: {total_chars/len(blog_articles):,.0f} chars/article")
        print(f"  Est. Tokens: {total_chars // 4:,}")
else:
    print(f"\n Blog file not found: {blog_file.name}")
    print(f"   Skipping blog ingestion")
    blog_articles = []

# STEP 2: SIMPLE FIXED-SIZE CHUNKING FOR BLOG ARTICLES (January 1st Approach)
if blog_articles:
    print(" FIXED-SIZE CHUNKING FOR BLOG ARTICLES")
    print("Strategy: Simple Fixed-Size with Overlap ")
    print(" Chunk Size: ~500 tokens (~2000 chars)")
    print(" Overlap: ~100 tokens (~400 chars)")
    print(" Simple, predictable, and reliable")
    print()
    
    # Initialize chunking strategy (no semantic chunking)
    print(f"Initializing fixed-size chunker...")
    chunking_strategy = create_chunking_strategy(
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        use_semantic=False  # Disable semantic chunking, use fixed-size
    )
    
    # Process all articles through fixed-size chunker
    all_chunks = []
    article_chunk_stats = []
    
    print(f"\n Processing {len(blog_articles)} articles...")
    
    for idx, article in enumerate(blog_articles):
        if not article.get('text'):
            continue
        
        try:
            # Apply simple fixed-size chunking (not semantic)
            chunks = chunking_strategy.chunk_blog_article(
                article=article,
                article_id=idx,
                use_semantic=False  # Force fixed-size
            )
            
            if chunks:
                all_chunks.extend(chunks)
                article_chunk_stats.append({
                    'title': article.get('title', 'Unknown')[:60],
                    'chunks': len(chunks),
                    'avg_tokens': sum(c.metadata.token_count for c in chunks) / len(chunks)
                })
            
            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1}/{len(blog_articles)} articles...")
        
        except Exception as e:
            print(f" Error processing article {idx}: {str(e)[:60]}")
            continue
    
    print(f"\n Chunking Complete!")
    print(f" Articles processed: {len(article_chunk_stats)}")
    print(f" Total chunks created: {len(all_chunks)}")
    if len(article_chunk_stats) > 0:
        print(f"   Avg chunks per article: {len(all_chunks)/len(article_chunk_stats):.1f}")
    
    # Show sample statistics
    if article_chunk_stats:
        print(f"\n Sample Chunk Statistics:")
        for i, stats in enumerate(article_chunk_stats[:3], 1):
            print(f"  [{i}] {stats['title']}...")
            print(f"      {stats['chunks']} chunks, {stats['avg_tokens']:.0f} avg tokens")
    
    # Validate chunk quality
    print(f"\n Validating Chunk Quality...")
    validation = validate_chunk_quality(all_chunks, verbose=False)
    print(f"   Token range: {validation['token_stats']['min']}-{validation['token_stats']['max']}")
    print(f"   Avg tokens: {validation['token_stats']['avg']:.1f}")
    if validation['season_mixing_detected'] == 0:
        print(f" Season mixing: 0 (Correct)")
    else:
        print(f" Season mixing: {validation['season_mixing_detected']}")
    
    # Prepare for vector database ingestion
    blog_documents = [chunk.content for chunk in all_chunks]
    blog_metadatas = [chunk.metadata.to_dict() for chunk in all_chunks]
    blog_ids = [chunk.metadata.chunk_id for chunk in all_chunks]
    
    print(f"\n Ingesting into ChromaDB...")
    print(f"   Total chunks to ingest: {len(blog_documents)}")
    
    # Embed and ingest in batches
    BLOG_BATCH_SIZE = 100
    total_batches = (len(blog_documents) + BLOG_BATCH_SIZE - 1) // BLOG_BATCH_SIZE
    
    blog_start = time.time()
    blog_embedded = 0
    
    for batch_idx in range(0, len(blog_documents), BLOG_BATCH_SIZE):
        batch_end = min(batch_idx + BLOG_BATCH_SIZE, len(blog_documents))
        batch_docs = blog_documents[batch_idx:batch_end]
        batch_metas = blog_metadatas[batch_idx:batch_end]
        batch_ids = blog_ids[batch_idx:batch_end]
        
        # Add to database (ChromaDB handles embedding internally)
        batch_start_time = time.time()
        vector_db.add_documents(batch_docs, batch_metas, batch_ids)
        batch_time = time.time() - batch_start_time
        
        blog_embedded += len(batch_docs)
        current_batch = (batch_idx // BLOG_BATCH_SIZE) + 1
        
        if current_batch % 3 == 0 or current_batch == total_batches:
            pct = (blog_embedded / len(blog_documents)) * 100
            print(f"   Batch {current_batch}/{total_batches}: {len(batch_docs)} chunks in {batch_time:.2f}s | Progress: {pct:.1f}%")
    
    blog_time = time.time() - blog_start

    print(f"\n BLOG ARTICLES INGESTION COMPLETE")
    print(f"   Articles Processed: {len(article_chunk_stats)}")
    print(f"   Fixed-Size Chunks Ingested: {len(blog_documents)}")
    print(f"   Time Taken: {blog_time:.2f}s ({blog_time/60:.2f} min)")
    print(f"   Speed: {len(blog_documents)/blog_time:.1f} chunks/s")
else:
    print("\n No blog articles to ingest")
    blog_time = 0
    article_chunk_stats = []
    blog_documents = []

# COMBINED STATS
print(f"\n{'-'*80}")
print(f" COMBINED INGESTION SUMMARY")
if 'df_to_embed' in dir():
    print(f"Player Records: {len(df_to_embed):,}")
else:
    print(f"Player Records: 0 (not yet ingested)")
print(f"Blog Articles: {len(article_chunk_stats):,}")
print(f"Blog Chunks: {len(blog_documents) if 'blog_documents' in dir() and blog_documents else 0:,}")

# Check actual DB count
final_db_count = vector_db.collection.count()
print(f"\n Total Documents in ChromaDB: {final_db_count:,}")


2026-01-05 13:53:52.162 | INFO     | src.chunking_strategy:__init__:127 - Initialized HybridChunkingStrategy (Fixed-Size: 500 tokens, Overlap: 100 tokens)


 LOADING TACTICAL BLOG ARTICLES

Loaded 36 tactical blog articles

Sample Article:
  Title: The Issue of Passivity – MX...
  Source: spielverlagerung.com
  Text Length: 24,918 characters

Blog Statistics:
  Total Articles: 36
  Total Characters: 968,427
  Avg Length: 26,901 chars/article
  Est. Tokens: 242,106
 FIXED-SIZE CHUNKING FOR BLOG ARTICLES
Strategy: Simple Fixed-Size with Overlap 
 Chunk Size: ~500 tokens (~2000 chars)
 Overlap: ~100 tokens (~400 chars)
 Simple, predictable, and reliable

Initializing fixed-size chunker...

 Processing 36 articles...


2026-01-05 13:53:52.442 | INFO     | src.chunking_strategy:chunk_blog_article:379 - ✓ Article 'The Issue of Passivity – MX' → 16 fixed-size chunks
2026-01-05 13:53:52.444 | INFO     | src.chunking_strategy:chunk_blog_article:379 - ✓ Article 'Frankfurt’s pressing adjustments secure victory ov' → 5 fixed-size chunks
2026-01-05 13:53:52.476 | INFO     | src.chunking_strategy:chunk_blog_article:379 - ✓ Article 'Chance Conversion – MH' → 45 fixed-size chunks
2026-01-05 13:53:52.477 | INFO     | src.chunking_strategy:chunk_blog_article:379 - ✓ Article 'U21 European Championship Final: English Manipulat' → 9 fixed-size chunks
2026-01-05 13:53:52.484 | INFO     | src.chunking_strategy:chunk_blog_article:379 - ✓ Article 'Tactical Theory: Diagonality' → 44 fixed-size chunks
2026-01-05 13:53:52.489 | INFO     | src.chunking_strategy:chunk_blog_article:379 - ✓ Article '(Mis)judgments in Football Analysis (1/3) – MH' → 26 fixed-size chunks
2026-01-05 13:53:52.494 | INFO     | src.chunking_strategy:

  Processed 10/36 articles...
  Processed 20/36 articles...
  Processed 30/36 articles...

 Chunking Complete!
 Articles processed: 36
 Total chunks created: 625
   Avg chunks per article: 17.4

 Sample Chunk Statistics:
  [1] The Issue of Passivity – MX...
      16 chunks, 419 avg tokens
  [2] Frankfurt’s pressing adjustments secure victory over Galatas...
      5 chunks, 406 avg tokens
  [3] Chance Conversion – MH...
      45 chunks, 424 avg tokens

 Validating Chunk Quality...
   Token range: 29-499
   Avg tokens: 419.4
 Season mixing: 0 (Correct)

 Ingesting into ChromaDB...
   Total chunks to ingest: 625


2026-01-05 13:54:02.236 | INFO     | src.database:add_documents:110 - Added 100 documents to collection
2026-01-05 13:54:09.216 | INFO     | src.database:add_documents:110 - Added 100 documents to collection
2026-01-05 13:54:16.334 | INFO     | src.database:add_documents:110 - Added 100 documents to collection


   Batch 3/7: 100 chunks in 7.12s | Progress: 48.0%


2026-01-05 13:54:26.063 | INFO     | src.database:add_documents:110 - Added 100 documents to collection
2026-01-05 13:54:34.435 | INFO     | src.database:add_documents:110 - Added 100 documents to collection
2026-01-05 13:54:41.788 | INFO     | src.database:add_documents:110 - Added 100 documents to collection


   Batch 6/7: 100 chunks in 7.46s | Progress: 96.0%


2026-01-05 13:54:44.240 | INFO     | src.database:add_documents:110 - Added 25 documents to collection


   Batch 7/7: 25 chunks in 2.35s | Progress: 100.0%

 BLOG ARTICLES INGESTION COMPLETE
   Articles Processed: 36
   Fixed-Size Chunks Ingested: 625
   Time Taken: 51.60s (0.86 min)
   Speed: 12.1 chunks/s

--------------------------------------------------------------------------------
 COMBINED INGESTION SUMMARY
Player Records: 13,888
Blog Articles: 36
Blog Chunks: 625

 Total Documents in ChromaDB: 70,904


## 9️⃣ Validate Semantic Search & Retrieval Quality

In [15]:
#  DUAL-SOURCE VALIDATION: Player Stats + Tactical Blogs
print(" VALIDATING DUAL-SOURCE RAG ARCHITECTURE")
print("-"*80)

# Check if vector_db exists (if not, provide instructions)
if 'vector_db' not in globals():
    print("\n ERROR: vector_db not defined!")
    print("\n SOLUTION: Run these cells first:")
    print("   1. Cell 1  - Imports & sys.path setup")
    print("   2. Cell 12 - ChromaDB initialization")
    print("   3. Then re-run this validation cell")
    raise NameError("vector_db not defined. Run Cell 1 and Cell 12 first.")

# Check database status
total_docs = vector_db.collection.count()
print(f"\n DATABASE STATUS:")
print(f"   Total documents: {total_docs:,}")
print(f"   Collection: {vector_db.collection_name}")

if total_docs == 0:
    print("\n Database is empty! Run ingestion cells first.")
else:
    print(f"\n Database ready for dual-source retrieval\n")
    
    # DIAGNOSTIC: Check data composition
    print("-"*80)
    print(" DATA COMPOSITION CHECK")
    print("-"*80)
    
    # Sample query to check what types of chunks exist
    sample_results = vector_db.query(query_text="football player", n_results=20)
    
    player_stat_chunks = 0
    blog_chunks = 0
    unknown_chunks = 0
    
    stat_modules_found = set()
    
    if sample_results and sample_results.get('metadatas'):
        for meta in sample_results['metadatas'][0]:
            chunk_type = meta.get('chunk_type', '')
            stat_module = meta.get('stat_module', '')
            
            if chunk_type == 'blog_article':
                blog_chunks += 1
            elif chunk_type == 'player_stats' or stat_module:
                player_stat_chunks += 1
                if stat_module:
                    stat_modules_found.add(stat_module)
            else:
                unknown_chunks += 1
    
    print(f"\nSample of 20 docs:")
    print(f"   Player stat chunks: {player_stat_chunks}/20")
    print(f"   Blog article chunks: {blog_chunks}/20")
    print(f"   Unknown: {unknown_chunks}/20")
    
    if stat_modules_found:
        print(f"\n   Stat modules detected: {', '.join(sorted(stat_modules_found))}")
    
    print("\n" + "-"*80)
    print(" QUERY TEST 1: PLAYER STATS (with stat_module filter)")
    print("-"*80)
    
    player_queries = [
        {
            "query": "young striker with high goal scoring",
            "description": " Young striker with goals",
            "expected_modules": ["identity", "shooting"]
        },
        {
            "query": "midfielder with excellent passing and assists",
            "description": " Midfielder passing & creativity",
            "expected_modules": ["identity", "passing"]
        },
        {
            "query": "defender with strong tackles and defensive actions",
            "description": " Defensive stats",
            "expected_modules": ["identity", "defensive"]
        }
    ]
    
    for i, query_config in enumerate(player_queries, 1):
        print(f"\n[Query {i}] {query_config['description']}")
        print(f"Search: '{query_config['query']}'")
        
        try:
            # Query with player_stats filter
            results = vector_db.query(
                query_text=query_config['query'],
                n_results=5,
                where={"chunk_type": "player_stats"}
            )
            
            if results and results.get('documents') and len(results['documents'][0]) > 0:
                print(f" Found {len(results['documents'][0])} player results")
                
                # Show top result
                top_meta = results['metadatas'][0][0]
                top_dist = results['distances'][0][0]
                similarity = 1 - top_dist
                
                player = top_meta.get('player', 'N/A')
                team = top_meta.get('team', 'N/A')
                season = top_meta.get('season', 'N/A')
                stat_module = top_meta.get('stat_module', 'N/A')
                
                print(f"   Top match ({similarity:.1%}):")
                print(f"      Player: {player}")
                print(f"      Team: {team} | Season: {season}")
                print(f"      Module: {stat_module}")
                print(f"      Metadata fields: {len(top_meta)}")
            else:
                print(f" No results found")
        
        except Exception as e:
            print(f"  Error: {str(e)[:80]}")
    
    print("\n" + "-"*80)
    print(" QUERY TEST 2: TACTICAL BLOGS (no filter - should get blogs)")
    print("-"*80)
    
    tactical_queries = [
        "pressing tactics and high defensive line",
        "attacking strategies in final third",
        "possession-based football philosophy"
    ]
    
    for i, query in enumerate(tactical_queries, 1):
        print(f"\n[Query {i}] '{query}'")
        
        try:
            results = vector_db.query(query_text=query, n_results=5)
            
            if results and results.get('documents') and len(results['documents'][0]) > 0:
                print(f"  Found {len(results['documents'][0])} results")
                
                # Count blog vs player chunks
                blog_count = 0
                player_count = 0
                
                for meta in results['metadatas'][0][:5]:
                    if meta.get('chunk_type') == 'blog_article':
                        blog_count += 1
                    elif meta.get('chunk_type') == 'player_stats' or meta.get('stat_module'):
                        player_count += 1
                
                print(f"    Blog articles: {blog_count}/5")
                print(f"    Player stats: {player_count}/5")
                
                # Show top blog article if found
                for j, (meta, dist) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
                    if meta.get('chunk_type') == 'blog_article':
                        similarity = 1 - dist
                        title = meta.get('article_title', meta.get('title', 'N/A'))
                        source = meta.get('source', 'N/A')
                        theme = meta.get('tactical_theme', 'N/A')
                        
                        print(f"\n  Top blog match ({similarity:.1%}):")
                        print(f"      Title: {title[:60]}...")
                        print(f"      Source: {source}")
                        if theme != 'N/A':
                            print(f" Theme: {theme}")
                        break
            else:
                print(f"    No results found")
        
        except Exception as e:
            print(f"   Error: {str(e)[:80]}")
    
    print("\n" + "-"*80)
    print(" QUERY TEST 3: COMBINED (player + tactical context)")
    
    combined_query = "creative midfielder with progressive passing and half-space attacking"
    print(f"\nQuery: '{combined_query}'")
    print("Expected: Mix of player stats (passing module) + tactical blog concepts")
    
    try:
        results = vector_db.query(query_text=combined_query, n_results=10)
        
        if results and results.get('documents'):
            player_chunks = 0
            blog_chunks = 0
            stat_modules = set()
            
            for meta in results['metadatas'][0]:
                if meta.get('chunk_type') == 'blog_article':
                    blog_chunks += 1
                elif meta.get('chunk_type') == 'player_stats' or meta.get('stat_module'):
                    player_chunks += 1
                    if meta.get('stat_module'):
                        stat_modules.add(meta.get('stat_module'))
            
            print(f"\n Mixed Results:")
            print(f"   Player stats: {player_chunks}/10")
            print(f"   Blog articles: {blog_chunks}/10")
            if stat_modules:
                print(f"   Stat modules: {', '.join(sorted(stat_modules))}")
            
            print(f"\n DUAL-SOURCE RETRIEVAL {' WORKING' if player_chunks > 0 and blog_chunks > 0 else ' CHECK NEEDED'}")
            
            if player_chunks > 0 and blog_chunks > 0:
                print("\n   SUCCESS: System can combine:")
                print("     Quantitative player stats (numbers, metrics)")
                print("     Qualitative tactical concepts (context, insights)")
                print("     Ready for rich scout report generation!")
        
    except Exception as e:
        print(f"  Error: {str(e)}")
    
    print("\n" + "-"*80)
    print(" FINAL SUMMARY")
    print("-"*80)
    print(f" Total documents: {total_docs:,}")
    print(f"    Expected: ~70,904 (70,279 player + 625 blog)")
    print(f"\n Dual-source architecture validated")
    print(f"    Player stats: Modular chunks (identity, shooting, passing, etc.)")
    print(f"    Tactical blogs: Fixed-size chunks with overlap")
    print(f"\n CRAG agent ready to:")
    print(f"  Retrieve player stats (quantitative)")
    print(f"  Retrieve tactical concepts (qualitative)")
    print(f"  Generate insights combining both")

 VALIDATING DUAL-SOURCE RAG ARCHITECTURE
--------------------------------------------------------------------------------

 DATABASE STATUS:
   Total documents: 70,904
   Collection: player_stats

 Database ready for dual-source retrieval

--------------------------------------------------------------------------------
 DATA COMPOSITION CHECK
--------------------------------------------------------------------------------

Sample of 20 docs:
   Player stat chunks: 20/20
   Blog article chunks: 0/20
   Unknown: 0/20

   Stat modules detected: defensive

--------------------------------------------------------------------------------
 QUERY TEST 1: PLAYER STATS (with stat_module filter)
--------------------------------------------------------------------------------

[Query 1]  Young striker with goals
Search: 'young striker with high goal scoring'


 Found 5 player results
   Top match (65.4%):
      Player: Olivier Giroud
      Team: Milan | Season: 2021-2022
      Module: shooting
      Metadata fields: 38

[Query 2]  Midfielder passing & creativity
Search: 'midfielder with excellent passing and assists'
 Found 5 player results
   Top match (69.6%):
      Player: Sergio Herrera
      Team: Osasuna | Season: 2021-2022
      Module: passing
      Metadata fields: 35

[Query 3]  Defensive stats
Search: 'defender with strong tackles and defensive actions'
 Found 5 player results
   Top match (66.0%):
      Player: Yıldırım Mert Çetin
      Team: Hellas Verona | Season: 2021-2022
      Module: defensive
      Metadata fields: 27

--------------------------------------------------------------------------------
 QUERY TEST 2: TACTICAL BLOGS (no filter - should get blogs)
--------------------------------------------------------------------------------

[Query 1] 'pressing tactics and high defensive line'
  Found 5 results
    Blog artic